# Layer 3 — Phase 4 v3: 보험 트리거 정량화 + 평가

## v2 → v3 변경사항 (7개)

### 필수 (버그 수정)
1. **★ `event_recall` 버그 수정** — `tp_events` → `len(matched_incidents)` (Recall > 1 가능성 제거)
2. **★ Lead Time `LOOKBACK_MIN`** — incident 시작 전 15분 이내 trigger도 detected로 인정

### 권장 (정확성/명확성)
3. **Primary/Fallback 기준 구조** — Recall ≥ 0.6 우선, 후보 없으면 0.5 fallback
4. **`false_alarm_rate` → `minute_false_alarm_rate`** 명명 (minute vs event 명확)
5. **`selected_trigger_config`에 final_performance 추가** (재현성)
6. **Y note 강화** — "parametric payout scenario" 명시
7. **(유지) Payout 50/80/100** — trigger 발동 자체가 심각 신호 → 50% 미만은 너무 후함

## 흐름 동일 (10 Steps)
```
Step A. Phase 3 결과 로드
Step B. θ × N 동시 grid search
Step C. Primary/Fallback 기준으로 (θ, N) 선택
Step D. 최종 L3 Trigger 생성
Step E. Event 단위 평가 (★ recall 버그 수정)
Step F. Lead Time 분석 (★ lookback 방식)
Step G. Y 시나리오 (note 강화)
Step H. Event-based Expected Payout
Step I. 결과 시각화
Step J. 7개 파일 저장 (final_performance 포함)
```

## Step A. 환경 셋업 + Part 3 결과 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = '/content/drive/MyDrive/layer3_data'
OUT = f'{ROOT}/processed'

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

!apt -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
scores = pd.read_parquet(os.path.join(OUT, 'cloud_risk_scores.parquet'))

with open(os.path.join(OUT, 'phase3_metadata.json'), 'r', encoding='utf-8') as f:
    p3_meta = json.load(f)
with open(os.path.join(OUT, 'phase2_metadata.json'), 'r', encoding='utf-8') as f:
    p2_meta = json.load(f)

print(f'scores shape: {scores.shape}')
print(f'incident 분: {scores["incident_flag"].sum()} / {len(scores)}')
display(scores.head())

In [ ]:
if scores['incident_flag'].sum() == 0:
    raise ValueError('incident 0. Part 1, 2 확인 필요.')
elif scores['incident_flag'].sum() < 10:
    print(f'⚠️ incident {scores["incident_flag"].sum()}분 — 평가 신뢰성 낮음')
else:
    print(f'✓ incident {scores["incident_flag"].sum()}분 — 평가 가능')

## Step B. θ × N grid search (v3: minute_false_alarm_rate 명명)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

THETA_GRID = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
N_GRID = [3, 5, 10, 15]

y_true = scores['incident_flag'].values
p_cloud = scores['P_cloud']

def make_trigger(p_cloud_series, theta, n):
    above = (p_cloud_series >= theta).astype(int)
    trigger = (above.rolling(window=n, min_periods=n).sum() == n).astype(int)
    return trigger

def count_trigger_events(trigger_series):
    starts = (trigger_series == 1) & (trigger_series.shift(1, fill_value=0) == 0)
    return int(starts.sum())

results = []
for theta in THETA_GRID:
    for n in N_GRID:
        trigger = make_trigger(p_cloud, theta, n)
        y_pred = trigger.values
        
        if y_pred.sum() == 0:
            results.append({
                'theta': theta, 'N': n,
                'minute_precision': 0.0, 'minute_recall': 0.0, 'minute_f1': 0.0,
                'trigger_minutes': 0, 'trigger_events': 0,
                'minute_false_alarm_rate': 0.0,
            })
            continue
        
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1v = f1_score(y_true, y_pred, zero_division=0)
        
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        minute_far = fp / max(fp + tn, 1)
        
        results.append({
            'theta': theta, 'N': n,
            'minute_precision': prec,
            'minute_recall': rec,
            'minute_f1': f1v,
            'trigger_minutes': int(y_pred.sum()),
            'trigger_events': count_trigger_events(trigger),
            'minute_false_alarm_rate': float(minute_far),
        })

results_df = pd.DataFrame(results)
print(f'총 {len(results_df)}개 (θ, N) 조합 평가 완료')
display(results_df.head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, title, cmap in [
    (axes[0, 0], 'minute_precision', 'Minute Precision', 'Greens'),
    (axes[0, 1], 'minute_recall', 'Minute Recall', 'Blues'),
    (axes[1, 0], 'minute_f1', 'Minute F1', 'Purples'),
    (axes[1, 1], 'minute_false_alarm_rate', 'Minute FAR', 'Reds'),
]:
    pivot = results_df.pivot(index='theta', columns='N', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap=cmap, ax=ax, cbar=True)
    ax.set_title(f'{title} (θ × N)')
    ax.set_xlabel('N (minutes)')
    ax.set_ylabel('θ')
plt.tight_layout()
plt.show()

## Step C. Primary/Fallback 기준으로 (θ, N) 선택 ⭐ v3

**v2 → v3 변경**: 단일 기준 (Recall ≥ 0.5) → **2단계 기준**

**선택 로직**:
1. **1차 (Primary)**: Recall ≥ 0.6, FAR ≤ 0.05 만족 후보 중 F1 최대
2. **2차 (Fallback)**: 1차 후보 없을 시 Recall ≥ 0.5로 완화
3. **3차 (Last resort)**: 그래도 없으면 전체 F1 최대

**보고서 멘트**: *"Recall 기준은 사고 미탐 최소화, FAR 기준은 오지급 위험 제한. 1차 기준 만족 후보가 없을 경우 단계적 완화 후 fallback."*

In [ ]:
PRIMARY_MIN_RECALL = 0.6
FALLBACK_MIN_RECALL = 0.5
MAX_FAR = 0.05

primary_feasible = results_df[
    (results_df['minute_recall'] >= PRIMARY_MIN_RECALL) &
    (results_df['minute_false_alarm_rate'] <= MAX_FAR) &
    (results_df['trigger_events'] > 0)
].copy()

fallback_feasible = results_df[
    (results_df['minute_recall'] >= FALLBACK_MIN_RECALL) &
    (results_df['minute_false_alarm_rate'] <= MAX_FAR) &
    (results_df['trigger_events'] > 0)
].copy()

print(f'Primary 기준 (Recall≥{PRIMARY_MIN_RECALL}, FAR≤{MAX_FAR}): {len(primary_feasible)}개')
print(f'Fallback 기준 (Recall≥{FALLBACK_MIN_RECALL}, FAR≤{MAX_FAR}): {len(fallback_feasible)}개')

if len(primary_feasible) > 0:
    best_row = primary_feasible.loc[primary_feasible['minute_f1'].idxmax()]
    SELECTION_TIER = 'primary'
    print(f'\n✓ 1차 기준 만족 후보 중 F1 최대 선택')
elif len(fallback_feasible) > 0:
    best_row = fallback_feasible.loc[fallback_feasible['minute_f1'].idxmax()]
    SELECTION_TIER = 'fallback'
    print(f'\n⚠️ 1차 후보 없음 — 2차 fallback 기준 사용')
else:
    best_row = results_df.loc[results_df['minute_f1'].idxmax()]
    SELECTION_TIER = 'last_resort'
    print(f'\n⚠️ 1, 2차 모두 없음 — 전체 F1 최대 사용')

THETA_FINAL = float(best_row['theta'])
N_FINAL = int(best_row['N'])

print(f'\n★ 최종 선택 (tier: {SELECTION_TIER}) ★')
print(f'   θ={THETA_FINAL:.2f}, N={N_FINAL}분')
print(f'   Minute Precision = {best_row["minute_precision"]:.4f}')
print(f'   Minute Recall    = {best_row["minute_recall"]:.4f}')
print(f'   Minute F1        = {best_row["minute_f1"]:.4f}')
print(f'   Minute FAR       = {best_row["minute_false_alarm_rate"]:.4f}')

## Step D. 최종 L3 Trigger 생성

In [ ]:
above_theta = (scores['P_cloud'] >= THETA_FINAL).astype(int)
trigger = (above_theta.rolling(window=N_FINAL, min_periods=N_FINAL).sum() == N_FINAL).astype(int)

trigger_event_start = (trigger == 1) & (trigger.shift(1, fill_value=0) == 0)
n_trigger_events = int(trigger_event_start.sum())

print(f'P_cloud ≥ θ 분: {above_theta.sum()}')
print(f'연속 N분 지속 trigger 분: {trigger.sum()}')
print(f'Trigger event 수: {n_trigger_events}회')

## Step E. Event 단위 평가 ⭐ v3 — recall 버그 수정

**v2 버그**: `event_recall = tp_events / len(incident_events)` — 하나의 incident에 trigger 여러 번 겹치면 **Recall > 1.0** 가능.

**v3 수정**: `event_recall = len(matched_incidents) / len(incident_events)` — matched_incidents set으로 중복 제거.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, precision_recall_curve, roc_curve, auc, roc_auc_score
)

def extract_events(binary_series):
    events = []
    in_event = False
    start = None
    prev_idx = None
    for idx, val in binary_series.items():
        if val == 1 and not in_event:
            start = idx
            in_event = True
        elif val == 0 and in_event:
            events.append((start, prev_idx))
            in_event = False
        prev_idx = idx
    if in_event:
        events.append((start, binary_series.index[-1]))
    return events

incident_events = extract_events(scores['incident_flag'])
trigger_events = extract_events(trigger)

print(f'Incident events: {len(incident_events)}회')
print(f'Trigger events:  {len(trigger_events)}회')

def events_overlap(e1, e2):
    return not (e1[1] < e2[0] or e2[1] < e1[0])

tp_events = 0
matched_incidents = set()
for ti, t_ev in enumerate(trigger_events):
    for ii, i_ev in enumerate(incident_events):
        if events_overlap(t_ev, i_ev):
            tp_events += 1
            matched_incidents.add(ii)
            break

fp_events = len(trigger_events) - tp_events
fn_events = len(incident_events) - len(matched_incidents)

# ★ v3 수정: matched_incidents 기준 recall
event_precision = tp_events / max(len(trigger_events), 1)
event_recall = len(matched_incidents) / max(len(incident_events), 1)   # ★ 버그 수정
event_f1 = 2 * event_precision * event_recall / max(event_precision + event_recall, 1e-9)

print(f'\n=== Event-level 평가 (v3: recall 버그 수정) ===')
print(f'  TP events: {tp_events}')
print(f'  Matched incidents (unique): {len(matched_incidents)}')
print(f'  FP events: {fp_events}')
print(f'  FN events: {fn_events}')
print(f'  Event Precision: {event_precision:.4f}')
print(f'  Event Recall:    {event_recall:.4f}  ← matched_incidents/total (Recall ≤ 1.0 보장)')
print(f'  Event F1:        {event_f1:.4f}')

# 검증
assert event_recall <= 1.0, f'Recall {event_recall} > 1.0 — 여전히 버그!'
print(f'\n✓ Recall ≤ 1.0 검증 통과')

In [ ]:
# Minute vs Event 비교
y_pred_minute = trigger.values
cm = confusion_matrix(y_true, y_pred_minute)
tn, fp, fn, tp = cm.ravel()

min_prec = precision_score(y_true, y_pred_minute, zero_division=0)
min_rec = recall_score(y_true, y_pred_minute, zero_division=0)
min_f1 = f1_score(y_true, y_pred_minute, zero_division=0)

y_score = scores['P_cloud'].values
auc_score = roc_auc_score(y_true, y_score)
precisions_curve, recalls_curve, _ = precision_recall_curve(y_true, y_score)
pr_auc = auc(recalls_curve, precisions_curve)

print('=== Minute-level vs Event-level 비교 ===')
compare = pd.DataFrame({
    'Minute-level': [min_prec, min_rec, min_f1],
    'Event-level':  [event_precision, event_recall, event_f1],
}, index=['Precision', 'Recall', 'F1'])
display(compare)

print(f'\nROC AUC: {auc_score:.4f}')
print(f'PR AUC:  {pr_auc:.4f}  ← class imbalance robust (발표 강조)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fpr, tpr, _ = roc_curve(y_true, y_score)
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc_score:.3f}')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(recalls_curve, precisions_curve, color='crimson', linewidth=2, label=f'PR AUC = {pr_auc:.3f}')
axes[1].scatter([min_rec], [min_prec], color='green', s=100, zorder=5,
                label=f'final minute (θ={THETA_FINAL:.2f}, N={N_FINAL})')
axes[1].scatter([event_recall], [event_precision], color='blue', s=100, marker='^', zorder=5,
                label=f'final event')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curve (발표 강조)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step F. Lead Time 분석 ⭐ v3 — lookback 방식

**v2 문제**: incident와 *"겹치는"* trigger만 detected로 봄. trigger가 incident 직전에 꺼지면 missed로 잘못 처리 → **조기 경보 성능 과소평가**.

**v3 해결**: `LOOKBACK_MIN=15`분. incident 시작 전 15분 이내 trigger도 detected로 인정.

**발표 멘트**: *"조기 경보 평가를 위해 incident 시작 전 15분 이내 trigger도 detected로 인정. Lead Time이 양수면 조기 경보 성공."*

In [ ]:
LOOKBACK_MIN = 15   # ★ v3: 조기 경보 lookback window

lead_time_records = []

for ii, (i_start, i_end) in enumerate(incident_events):
    lookback_start = i_start - pd.Timedelta(minutes=LOOKBACK_MIN)
    
    # ★ v3: 겹침 OR lookback window 내 trigger
    candidate_triggers = [
        (ti, t_start, t_end) 
        for ti, (t_start, t_end) in enumerate(trigger_events)
        if (
            # 1. 겹치는 trigger
            events_overlap((t_start, t_end), (i_start, i_end))
            # 2. incident 시작 전 lookback window 내 발생 trigger
            or (t_start >= lookback_start and t_start <= i_start)
        )
    ]
    
    if candidate_triggers:
        ti, t_start, t_end = min(candidate_triggers, key=lambda x: x[1])
        lead_minutes = (i_start - t_start).total_seconds() / 60
        status = 'detected'
    else:
        ti, t_start, lead_minutes = None, None, None
        status = 'missed'
    
    lead_time_records.append({
        'incident_idx': ii,
        'incident_start': i_start,
        'incident_end': i_end,
        'incident_duration_min': int((i_end - i_start).total_seconds() / 60) + 1,
        'trigger_start': t_start,
        'lead_time_min': lead_minutes,
        'status': status,
    })

lead_time_df = pd.DataFrame(lead_time_records)

print(f'Lead Time 분석 (lookback={LOOKBACK_MIN}분):')
print(f'  Incident events: {len(lead_time_df)}회')
print(f'  Detected: {(lead_time_df["status"] == "detected").sum()}회')
print(f'  Missed:   {(lead_time_df["status"] == "missed").sum()}회')

if (lead_time_df['status'] == 'detected').any():
    detected = lead_time_df[lead_time_df['status'] == 'detected']
    print(f'\n=== Lead Time 통계 (detected만) ===')
    print(f'  평균: {detected["lead_time_min"].mean():+.1f} 분')
    print(f'  중앙값: {detected["lead_time_min"].median():+.1f} 분')
    print(f'  범위: {detected["lead_time_min"].min():+.0f} ~ {detected["lead_time_min"].max():+.0f} 분')
    
    n_early = (detected['lead_time_min'] > 0).sum()
    n_simul = (detected['lead_time_min'] == 0).sum()
    n_delayed = (detected['lead_time_min'] < 0).sum()
    print(f'\n  조기 경보 (>0): {n_early}회')
    print(f'  동시 발동 (=0): {n_simul}회')
    print(f'  Detection delay (<0): {n_delayed}회')
    print(f'  ★ 조기 경보 비율: {n_early/len(detected)*100:.1f}%')

display(lead_time_df)

In [ ]:
if (lead_time_df['status'] == 'detected').any():
    detected = lead_time_df[lead_time_df['status'] == 'detected']
    
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['green' if x > 0 else ('orange' if x == 0 else 'red') for x in detected['lead_time_min']]
    ax.bar(range(len(detected)), detected['lead_time_min'], color=colors, alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axhline(y=LOOKBACK_MIN, color='gray', linestyle=':', alpha=0.5, label=f'lookback={LOOKBACK_MIN}분')
    ax.set_xlabel('Incident #')
    ax.set_ylabel('Lead Time (분)')
    ax.set_title(f'Lead Time per Incident (lookback={LOOKBACK_MIN}분)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Step G. Y 시나리오 (v3: note 강화)

In [ ]:
ANNUAL_REVENUE_KRW = 1_400_000_000_000
MFG_COST_RATIO = 0.60
AI_PROCESS_REVENUE_RATIO = 0.10
ANNUAL_INCIDENTS_BASELINE = 8
PAYOUT_RATIO = 0.30

ai_dependent_cost = ANNUAL_REVENUE_KRW * MFG_COST_RATIO * AI_PROCESS_REVENUE_RATIO
per_incident_loss = ai_dependent_cost / ANNUAL_INCIDENTS_BASELINE
Y_FINAL = per_incident_loss * PAYOUT_RATIO

print('=== Y 산정 (Parametric Payout Scenario) ===')
print('⚠️ 본 연구의 Y는 실제 보험료 산정값이 아닌 parametric payout scenario.')
print('   실제 상품화 시 보험사 손해율 데이터 + 기업별 손실 데이터로 보정 필요.')
print(f'\n  연간 매출: {ANNUAL_REVENUE_KRW/1e8:,.0f}억원 (가정)')
print(f'  제조원가율: {MFG_COST_RATIO*100:.0f}%')
print(f'  AI 공정 의존 매출: {AI_PROCESS_REVENUE_RATIO*100:.0f}%')
print(f'  연간 incident: {ANNUAL_INCIDENTS_BASELINE}회')
print(f'  보장 비율: {PAYOUT_RATIO*100:.0f}%')
print(f'\n  ★ Y = {Y_FINAL/1e8:,.2f}억원/event ★')

In [ ]:
# Tiered payout (현재 50/80/100 유지)
# v3 정당화: trigger 발동 자체가 "심각" 신호 — 최소 50% 보장이 일관적
tiered_payout = pd.DataFrame([
    {'level': 'Level 1 (경미)', 'P_cloud_range': f'[{THETA_FINAL:.2f}, 0.85)', 'payout_ratio': 0.5, 'payout_krw': Y_FINAL * 0.5},
    {'level': 'Level 2 (중대)', 'P_cloud_range': '[0.85, 0.95)', 'payout_ratio': 0.8, 'payout_krw': Y_FINAL * 0.8},
    {'level': 'Level 3 (심각)', 'P_cloud_range': '[0.95, 1.00]', 'payout_ratio': 1.0, 'payout_krw': Y_FINAL * 1.0},
])
tiered_payout['payout_billion_krw'] = tiered_payout['payout_krw'] / 1e8

print('=== Tiered Payout (Y_FINAL × ratio 적용) ===')
display(tiered_payout)
print('정당화: Trigger 발동 자체가 N분 연속 임계 초과 — 최소 50% 보장이 일관성 있음')

## Step H. Event-based Expected Payout

In [ ]:
n_days = (scores.index.max() - scores.index.min()).days + 1
DAYS_PER_YEAR = 365

trigger_events_per_day = n_trigger_events / max(n_days, 1)
trigger_events_per_year = trigger_events_per_day * DAYS_PER_YEAR

tp_events_per_year = (tp_events / max(n_days, 1)) * DAYS_PER_YEAR
fp_events_per_year = (fp_events / max(n_days, 1)) * DAYS_PER_YEAR

expected_payout_per_year = trigger_events_per_year * Y_FINAL
fp_loss_per_year = fp_events_per_year * Y_FINAL

print('=== Event-based 연간 환산 ===')
print(f'데이터 기간: {n_days}일')
print(f'\n연간 Trigger event: {trigger_events_per_year:.1f}회')
print(f'  └ TP: {tp_events_per_year:.1f}회')
print(f'  └ FP: {fp_events_per_year:.1f}회 (오지급)')
print(f'\n★ 연간 예상 보험금 지급: {expected_payout_per_year/1e8:,.2f}억원')
print(f'   └ 오지급 손실: {fp_loss_per_year/1e8:,.2f}억원')

## Step I. 결과 시각화

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(scores.index, scores['P_cloud'], linewidth=0.7, color='steelblue', label='P_cloud')
axes[0].axhline(y=THETA_FINAL, color='red', linestyle='--', alpha=0.5, label=f'θ={THETA_FINAL:.2f}')
axes[0].set_title('① Cloud Risk Score with Threshold')
axes[0].set_ylabel('P_cloud')
axes[0].legend(loc='upper right')

axes[1].fill_between(scores.index, 0, trigger, color='crimson', alpha=0.6)
axes[1].set_title(f'② Insurance Trigger ({n_trigger_events} events)')
axes[1].set_ylabel('trigger')
axes[1].set_yticks([0, 1])

axes[2].fill_between(scores.index, 0, scores['incident_flag'], color='darkorange', alpha=0.5)
axes[2].set_title(f'③ Incident ({len(incident_events)} events)')
axes[2].set_ylabel('incident')
axes[2].set_yticks([0, 1])

if (lead_time_df['status'] == 'detected').any():
    for _, row in lead_time_df.iterrows():
        if row['status'] == 'detected':
            color = 'green' if row['lead_time_min'] > 0 else ('orange' if row['lead_time_min'] == 0 else 'red')
            axes[3].axvline(x=row['incident_start'], color='red', alpha=0.3, linestyle='--')
            if row['trigger_start'] is not None:
                axes[3].axvline(x=row['trigger_start'], color=color, alpha=0.7)
        else:
            axes[3].axvline(x=row['incident_start'], color='gray', alpha=0.5, linestyle=':')

axes[3].set_title(f'④ Lead Time (lookback={LOOKBACK_MIN}분, green=early, orange=simultaneous, red=delayed)')
axes[3].set_xlabel('Time')
axes[3].set_yticks([])

plt.tight_layout()
plt.show()

## Step J. 7개 파일 저장 (v3: selected_trigger_config에 성능 추가)

In [ ]:
# 1. trigger_evaluation_results.csv
results_df.to_csv(os.path.join(OUT, 'trigger_evaluation_results.csv'), index=False)
print(f'저장: trigger_evaluation_results.csv')

# Lead Time 평균 (metadata용)
lead_time_mean = None
lead_time_median = None
if (lead_time_df['status'] == 'detected').any():
    detected_df = lead_time_df[lead_time_df['status'] == 'detected']
    lead_time_mean = float(detected_df['lead_time_min'].mean())
    lead_time_median = float(detected_df['lead_time_min'].median())

# 2. selected_trigger_config.json (v3: final_performance 추가)
trigger_config = {
    'theta': THETA_FINAL,
    'N_minutes': N_FINAL,
    'Y_payout_krw': float(Y_FINAL),
    'selection_criteria': {
        'primary_min_recall': PRIMARY_MIN_RECALL,
        'fallback_min_recall': FALLBACK_MIN_RECALL,
        'max_minute_far': MAX_FAR,
        'tiebreaker': 'minute F1 maximum',
        'tier_used': SELECTION_TIER,
    },
    'final_formula': f'IF P_cloud >= {THETA_FINAL:.2f} for >= {N_FINAL} consecutive minutes THEN payout = {Y_FINAL/1e8:.2f}억원',
    # ★ v3 추가: final_performance
    'final_performance': {
        'minute_precision': float(min_prec),
        'minute_recall': float(min_rec),
        'minute_f1': float(min_f1),
        'minute_false_alarm_rate': float(best_row['minute_false_alarm_rate']),
        'event_precision': float(event_precision),
        'event_recall': float(event_recall),
        'event_f1': float(event_f1),
        'roc_auc': float(auc_score),
        'pr_auc': float(pr_auc),
        'lead_time_mean_min': lead_time_mean,
        'lead_time_median_min': lead_time_median,
        'lookback_min': LOOKBACK_MIN,
    },
}
with open(os.path.join(OUT, 'selected_trigger_config.json'), 'w', encoding='utf-8') as f:
    json.dump(trigger_config, f, ensure_ascii=False, indent=2)
print(f'저장: selected_trigger_config.json (with final_performance)')

# 3. insurance_trigger.parquet
trigger_table = pd.DataFrame({
    'P_cloud': scores['P_cloud'],
    'above_theta': above_theta,
    'trigger': trigger,
    'trigger_event_start': trigger_event_start.astype(int),
    'incident_flag': scores['incident_flag'],
}, index=scores.index)
trigger_table.to_parquet(os.path.join(OUT, 'insurance_trigger.parquet'))
print(f'저장: insurance_trigger.parquet')

In [ ]:
# 4. ★ layer3_final_output.parquet ★
def assign_trigger_level(p_cloud_val, trigger_val):
    if trigger_val == 0:
        return 0
    elif p_cloud_val >= 0.95:
        return 3
    elif p_cloud_val >= 0.85:
        return 2
    else:
        return 1

trigger_level = pd.Series([
    assign_trigger_level(p, t) 
    for p, t in zip(scores['P_cloud'].values, trigger.values)
], index=scores.index)

payout_rate_map = {0: 0.0, 1: 0.5, 2: 0.8, 3: 1.0}
payout_rate = trigger_level.map(payout_rate_map)

layer3_final = pd.DataFrame({
    'P_cloud': scores['P_cloud'],
    'P_cloud_rule': scores['P_cloud_rule'],
    'P_cloud_lstm': scores['P_cloud_lstm'],
    'Monitoring_Failure_Probability': scores['Monitoring_Failure_Probability'],
    'AI_Decision_Reliability_Penalty': scores['AI_Decision_Reliability_Penalty'],
    'incident_flag': scores['incident_flag'],
    'L3_trigger': trigger,
    'trigger_level': trigger_level,
    'payout_rate_Y': payout_rate,
}, index=scores.index)
layer3_final.index.name = 'timestamp'
layer3_final.to_parquet(os.path.join(OUT, 'layer3_final_output.parquet'))
print(f'★ 저장: layer3_final_output.parquet')

# 5. evaluation_metrics.json
eval_metrics = {
    'minute_level': {
        'precision': float(min_prec),
        'recall': float(min_rec),
        'f1': float(min_f1),
        'false_alarm_rate': float(best_row['minute_false_alarm_rate']),
        'confusion_matrix': {'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)},
    },
    'event_level': {
        'precision': float(event_precision),
        'recall': float(event_recall),    # ★ v3 수정 버전
        'f1': float(event_f1),
        'tp_events': int(tp_events),
        'fp_events': int(fp_events),
        'fn_events': int(fn_events),
        'matched_incidents': int(len(matched_incidents)),
        'note': 'recall = matched_incidents / total incidents (bug-fixed)',
    },
    'auc': {
        'roc_auc': float(auc_score),
        'pr_auc': float(pr_auc),
    },
    'lead_time': {
        'lookback_min': LOOKBACK_MIN,
        'mean': lead_time_mean,
        'median': lead_time_median,
        'n_detected': int((lead_time_df['status'] == 'detected').sum()),
        'n_missed': int((lead_time_df['status'] == 'missed').sum()),
        'n_early_warning': int(((lead_time_df['status'] == 'detected') & (lead_time_df['lead_time_min'] > 0)).sum()),
    },
}
with open(os.path.join(OUT, 'evaluation_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)
print(f'저장: evaluation_metrics.json')

# 6. insurance_pricing.json (v3: note 강화)
pricing = {
    'note': (
        '본 연구의 Y는 실제 보험료 산정값이 아닌 parametric payout scenario. '
        '실제 상품화 시 보험사 손해율 데이터와 기업별 생산 손실 데이터로 보정 필요. '
        'All values are illustrative assumptions, not actual insurance pricing.'
    ),
    'theta': float(THETA_FINAL),
    'N_minutes': int(N_FINAL),
    'Y_payout_krw': float(Y_FINAL),
    'Y_payout_in_billion_krw': float(Y_FINAL / 1e8),
    'assumptions': {
        'annual_revenue_krw': ANNUAL_REVENUE_KRW,
        'mfg_cost_ratio': MFG_COST_RATIO,
        'ai_process_revenue_ratio': AI_PROCESS_REVENUE_RATIO,
        'annual_incidents_baseline': ANNUAL_INCIDENTS_BASELINE,
        'payout_ratio': PAYOUT_RATIO,
    },
    'tiered_payout': tiered_payout.to_dict('records'),
    'annual_simulation_event_based': {
        'data_period_days': int(n_days),
        'trigger_events_per_year': float(trigger_events_per_year),
        'tp_events_per_year': float(tp_events_per_year),
        'fp_events_per_year': float(fp_events_per_year),
        'expected_payout_per_year_krw': float(expected_payout_per_year),
        'fp_loss_per_year_krw': float(fp_loss_per_year),
    },
}
with open(os.path.join(OUT, 'insurance_pricing.json'), 'w', encoding='utf-8') as f:
    json.dump(pricing, f, ensure_ascii=False, indent=2)
print(f'저장: insurance_pricing.json')

# 7. phase4_metadata.json
p4_metadata = {
    'phase4_summary': {
        'version': 'v3',
        'final_theta': float(THETA_FINAL),
        'final_N_minutes': int(N_FINAL),
        'final_Y_krw': float(Y_FINAL),
        'trigger_formula': f'IF P_cloud >= {THETA_FINAL:.2f} for >= {N_FINAL} min THEN payout = {Y_FINAL/1e8:.2f}억원',
        'selection_tier': SELECTION_TIER,
    },
    'grid_search': {
        'theta_grid': THETA_GRID,
        'N_grid': N_GRID,
        'total_combinations': len(results_df),
    },
    'evaluation': eval_metrics,
    'pricing': pricing,
    'v3_changes': {
        '1_event_recall_bug': 'matched_incidents 기준으로 수정 (Recall ≤ 1.0 보장)',
        '2_lead_time_lookback': f'LOOKBACK_MIN={LOOKBACK_MIN}분 — 조기 경보 평가 정확',
        '3_primary_fallback': f'Primary Recall≥{PRIMARY_MIN_RECALL}, Fallback Recall≥{FALLBACK_MIN_RECALL}',
        '4_naming': 'false_alarm_rate → minute_false_alarm_rate',
        '5_final_performance': 'selected_trigger_config에 성능 지표 추가',
        '6_y_note': 'parametric payout scenario 명시 강화',
        '7_payout_tier': '50/80/100 유지 (trigger 발동 자체가 심각 신호)',
    },
}
with open(os.path.join(OUT, 'phase4_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(p4_metadata, f, ensure_ascii=False, indent=2)
print(f'저장: phase4_metadata.json')

print('\n=== Part 4 v3 완료 ===')
print(f'★ 최종 트리거: P_cloud >= {THETA_FINAL:.2f} for ≥ {N_FINAL}분 → {Y_FINAL/1e8:.2f}억원')
print(f'★ Minute: F1={min_f1:.3f}, P={min_prec:.3f}, R={min_rec:.3f}, FAR={best_row["minute_false_alarm_rate"]:.3f}')
print(f'★ Event:  F1={event_f1:.3f}, P={event_precision:.3f}, R={event_recall:.3f} (≤1.0 검증)')
print(f'★ AUC: ROC={auc_score:.3f}, PR={pr_auc:.3f}')
if lead_time_mean is not None:
    print(f'★ Lead Time (lookback {LOOKBACK_MIN}분): 평균 {lead_time_mean:+.1f}분')

## ✅ Part 4 v3 완료 체크리스트

### v2 → v3 변경 (7개)
- [x] **1. event_recall 버그 수정** — matched_incidents 기준
- [x] **2. Lead Time LOOKBACK_MIN=15분** — 조기 경보 정확
- [x] 3. Primary (Recall≥0.6) / Fallback (Recall≥0.5) 기준 구조
- [x] 4. `false_alarm_rate` → `minute_false_alarm_rate` 명명
- [x] 5. `selected_trigger_config`에 `final_performance` 추가
- [x] 6. Y "parametric payout scenario" 표현 강화
- [x] 7. Payout 50/80/100 유지 (정당화 강화)

## 발표 방어 멘트 (v3 강화)

| 질문 | 답변 |
|---|---|
| "Event Recall이 1을 넘을 수 있나?" | "v3에서 matched_incidents (unique) 기준으로 수정 → Recall ≤ 1 보장" |
| "Lead Time 어떻게 정의?" | "incident 시작 전 15분 이내 trigger도 detected로 인정 — 조기 경보 정확 평가" |
| "Recall 0.6 vs 0.5?" | "Primary 0.6 우선 시도, 후보 없으면 fallback 0.5로 완화 — 데이터 적응성" |
| "FAR 단위?" | "minute_false_alarm_rate 명시 — event 단위 FAR과 명확히 구분" |
| "Y 정확한 값?" | "parametric payout scenario. 실제 상품화 시 보험사 손해율 데이터 + 기업별 손실로 보정" |
| "50% Level 1 너무 높지 않나?" | "Trigger 발동 자체가 N분 연속 임계 초과 — 최소 50% 보장이 일관성" |